In [ ]:
import numpy as np
import pandas as pd


# ============================================================
# 1) COMPONENT PLACEMENT MODEL
# ============================================================
def geometric_component_shares(num_components: int, ratio: float) -> np.ndarray:
    """
    Create a normalized geometric distribution for component placement.

    p_i = (1-r) r^(i-1) / (1-r^K),  for i = 1..K

    Parameters
    ----------
    num_components : int
        Number of sequential components K.
    ratio : float
        Geometric ratio r, must satisfy 0 < r < 1.

    Returns
    -------
    np.ndarray
        Array of length K containing p_1, p_2, ..., p_K.
    """
    if not (0 < ratio < 1):
        raise ValueError("ratio must satisfy 0 < r < 1")

    component_shares = np.array(
        [
            (1 - ratio) * (ratio**i) / (1 - ratio**num_components)
            for i in range(num_components)
        ],
        dtype=float,
    )
    return component_shares


def allocate_hosts(total_devices: int, component_shares: np.ndarray) -> np.ndarray:
    """
    Convert fractional placement shares into integer host counts.

    We round N * p_i to integers and then adjust to ensure sum(n_i) = N.

    Parameters
    ----------
    total_devices : int
        Total number of devices N.
    component_shares : np.ndarray
        Placement shares p_i.

    Returns
    -------
    np.ndarray
        Integer host counts n_i for each component.
    """
    raw_counts = np.rint(total_devices * component_shares).astype(int)

    # Ensure at least one host per component
    raw_counts = np.maximum(raw_counts, 1)

    # Fix total so that sum(n_i) = N
    difference = total_devices - raw_counts.sum()

    # If we need to add hosts, add them to components with larger p_i first
    # If we need to remove hosts, remove from larger p_i first, but keep counts >= 1
    order = np.argsort(-component_shares)

    index = 0
    while difference != 0:
        component_index = order[index % len(order)]

        if difference > 0:
            raw_counts[component_index] += 1
            difference -= 1
        else:
            if raw_counts[component_index] > 1:
                raw_counts[component_index] -= 1
                difference += 1

        index += 1

    return raw_counts


# ============================================================
# 2) ANALYTICAL FORMULAS
# ============================================================
def compute_theoretical_metrics(
    total_devices: int,
    num_components: int,
    geometric_ratio: float,
    arrival_rate: float,
    service_rates: np.ndarray,
    source_delay: float,
    inter_component_delays: np.ndarray,
):
    """
    Compute theoretical values for the split-execution model.

    Returns component-wise and end-to-end analytical metrics.
    """
    component_shares = geometric_component_shares(num_components, geometric_ratio)
    host_counts = allocate_hosts(total_devices, component_shares)

    arrival_rate_per_host = arrival_rate / host_counts
    utilization = arrival_rate_per_host / service_rates

    # M/M/1 theory
    sojourn_time = 1.0 / (service_rates - arrival_rate_per_host)  # W_i
    queue_wait_time = sojourn_time - (1.0 / service_rates)  # W_q,i

    avg_number_in_system = utilization / (1.0 - utilization)  # L_i
    avg_queue_length = avg_number_in_system - utilization  # L_q,i

    total_queue_delay = queue_wait_time.sum()
    total_service_time = np.sum(1.0 / service_rates)
    total_comm_delay = source_delay + inter_component_delays.sum()

    total_end_to_end_delay = total_queue_delay + total_service_time + total_comm_delay

    return {
        "component_shares": component_shares,
        "host_counts": host_counts,
        "arrival_rate_per_host": arrival_rate_per_host,
        "utilization": utilization,
        "sojourn_time": sojourn_time,
        "queue_wait_time": queue_wait_time,
        "avg_number_in_system": avg_number_in_system,
        "avg_queue_length": avg_queue_length,
        "total_queue_delay": total_queue_delay,
        "total_service_time": total_service_time,
        "total_comm_delay": total_comm_delay,
        "total_end_to_end_delay": total_end_to_end_delay,
    }


# ============================================================
# 3) M/M/1 SIMULATION FOR ONE COMPONENT
# ============================================================
def simulate_mm1_queue(
    arrival_rate: float,
    service_rate: float,
    num_jobs: int = 50000,
    warmup_jobs: int = 5000,
    random_seed: int = 1,
):
    """
    Simulate one M/M/1 queue using event times.

    We generate:
    - exponential inter-arrival times with rate arrival_rate
    - exponential service times with rate service_rate

    For each job, we compute:
    - waiting time
    - system time (waiting + service)

    Then we estimate:
    - W (mean system time)
    - W_q (mean waiting time)
    - L using Little's law: L = lambda * W

    Parameters
    ----------
    arrival_rate : float
        Poisson arrival rate lambda.
    service_rate : float
        Exponential service rate mu.
    num_jobs : int
        Total number of jobs to simulate.
    warmup_jobs : int
        Initial jobs discarded to reduce transient bias.
    random_seed : int
        Seed for reproducibility.

    Returns
    -------
    dict
        Simulated W, W_q, and L.
    """
    if arrival_rate >= service_rate:
        raise ValueError(
            "Queue is unstable because arrival_rate must be < service_rate"
        )

    rng = np.random.default_rng(random_seed)

    arrival_times = np.cumsum(rng.exponential(scale=1.0 / arrival_rate, size=num_jobs))
    service_times = rng.exponential(scale=1.0 / service_rate, size=num_jobs)

    start_service_times = np.zeros(num_jobs)
    departure_times = np.zeros(num_jobs)
    waiting_times = np.zeros(num_jobs)
    system_times = np.zeros(num_jobs)

    server_available_time = 0.0

    for i in range(num_jobs):
        start_service_times[i] = max(arrival_times[i], server_available_time)
        waiting_times[i] = start_service_times[i] - arrival_times[i]
        departure_times[i] = start_service_times[i] + service_times[i]
        system_times[i] = departure_times[i] - arrival_times[i]
        server_available_time = departure_times[i]

    # Drop warm-up jobs
    effective_waiting_times = waiting_times[warmup_jobs:]
    effective_system_times = system_times[warmup_jobs:]

    mean_waiting_time = np.mean(effective_waiting_times)
    mean_system_time = np.mean(effective_system_times)

    # Little's law estimate
    mean_number_in_system = arrival_rate * mean_system_time
    mean_queue_length = arrival_rate * mean_waiting_time

    return {
        "sim_W": mean_system_time,
        "sim_Wq": mean_waiting_time,
        "sim_L": mean_number_in_system,
        "sim_Lq": mean_queue_length,
    }


# ============================================================
# 4) FULL EVALUATION: THEORY vs SIMULATION
# ============================================================
def run_evaluation():
    """
    Run the complete evaluation for all components and print a comparison table.
    """
    # -----------------------------
    # Model parameters
    # -----------------------------
    total_devices = 100
    num_components = 5
    geometric_ratio = 0.75
    global_arrival_rate = 8.0

    # Service rates of each component host
    service_rates = np.array([1.2, 1.2, 1.2, 1.2, 1.2], dtype=float)

    # Communication delays
    source_to_first_component_delay = 1.0
    inter_component_delays = np.array([1.0, 1.0, 1.0, 1.0], dtype=float)

    # -----------------------------
    # Theoretical values
    # -----------------------------
    theory = compute_theoretical_metrics(
        total_devices=total_devices,
        num_components=num_components,
        geometric_ratio=geometric_ratio,
        arrival_rate=global_arrival_rate,
        service_rates=service_rates,
        source_delay=source_to_first_component_delay,
        inter_component_delays=inter_component_delays,
    )

    # -----------------------------
    # Simulated values for each component
    # -----------------------------
    simulated_W = []
    simulated_Wq = []
    simulated_L = []
    simulated_Lq = []

    for idx in range(num_components):
        sim_result = simulate_mm1_queue(
            arrival_rate=theory["arrival_rate_per_host"][idx],
            service_rate=service_rates[idx],
            num_jobs=50000,
            warmup_jobs=5000,
            random_seed=idx + 1,
        )
        simulated_W.append(sim_result["sim_W"])
        simulated_Wq.append(sim_result["sim_Wq"])
        simulated_L.append(sim_result["sim_L"])
        simulated_Lq.append(sim_result["sim_Lq"])

    simulated_W = np.array(simulated_W)
    simulated_Wq = np.array(simulated_Wq)
    simulated_L = np.array(simulated_L)
    simulated_Lq = np.array(simulated_Lq)

    # -----------------------------
    # End-to-end simulation values
    # -----------------------------
    simulated_total_queue_delay = simulated_Wq.sum()
    simulated_total_service_time = np.sum(1.0 / service_rates)
    simulated_total_comm_delay = (
        source_to_first_component_delay + inter_component_delays.sum()
    )
    simulated_total_e2e_delay = (
        simulated_total_queue_delay
        + simulated_total_service_time
        + simulated_total_comm_delay
    )

    # -----------------------------
    # Prepare comparison table
    # -----------------------------
    results = pd.DataFrame(
        {
            "Component": [f"C{i+1}" for i in range(num_components)],
            "Hosts n_i": theory["host_counts"],
            "Theory W_i": theory["sojourn_time"],
            "Sim W_i": simulated_W,
            "Theory Wq_i": theory["queue_wait_time"],
            "Sim Wq_i": simulated_Wq,
            "Theory L_i": theory["avg_number_in_system"],
            "Sim L_i": simulated_L,
            "Theory Lq_i": theory["avg_queue_length"],
            "Sim Lq_i": simulated_Lq,
        }
    )

    pd.set_option("display.float_format", lambda x: f"{x:.4f}")

    print("\n=== Component-wise Comparison ===\n")
    print(results.to_string(index=False))

    print("\n=== End-to-End Delay ===")
    print(f"Theory  T_e2e = {theory['total_end_to_end_delay']:.4f}")
    print(f"Sim     T_e2e = {simulated_total_e2e_delay:.4f}")

    print("\n=== Delay Breakdown ===")
    print(f"Theory  Queue Delay   = {theory['total_queue_delay']:.4f}")
    print(f"Sim     Queue Delay   = {simulated_total_queue_delay:.4f}")
    print(f"Theory  Service Time  = {theory['total_service_time']:.4f}")
    print(f"Sim     Service Time  = {simulated_total_service_time:.4f}")
    print(f"Theory  Comm Delay    = {theory['total_comm_delay']:.4f}")
    print(f"Sim     Comm Delay    = {simulated_total_comm_delay:.4f}")


if __name__ == "__main__":
    run_evaluation()


=== Component-wise Comparison ===

Component  Hosts n_i  Theory W_i  Sim W_i  Theory Wq_i  Sim Wq_i  Theory L_i  Sim L_i  Theory Lq_i  Sim Lq_i
       C1         33      1.0443   1.0392       0.2110    0.2113      0.2532   0.2519       0.0511    0.0512
       C2         25      1.1364   1.1395       0.3030    0.3053      0.3636   0.3646       0.0970    0.0977
       C3         18      1.3235   1.3296       0.4902    0.4939      0.5882   0.5909       0.2179    0.2195
       C4         14      1.5909   1.5613       0.7576    0.7352      0.9091   0.8922       0.4329    0.4201
       C5         10      2.5000   2.4742       1.6667    1.6425      2.0000   1.9794       1.3333    1.3140

=== End-to-End Delay ===
Theory  T_e2e = 12.5951
Sim     T_e2e = 12.5548

=== Delay Breakdown ===
Theory  Queue Delay   = 3.4284
Sim     Queue Delay   = 3.3881
Theory  Service Time  = 4.1667
Sim     Service Time  = 4.1667
Theory  Comm Delay    = 5.0000
Sim     Comm Delay    = 5.0000
